### Langchain + Typsense + Groq LLM + RAG Application

##### TXT Files → TextLoader → Chunking → HuggingFace Embeddings → Typesense Vector DB → Retriever → Groq LLM → RAG Answer

In [1]:
### Langchain + Typsense + Groq LLM + RAG Application

import typesense
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader , DirectoryLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings 

c:\Users\acer\Documents\Typesense rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
#Loading 
loader1 = TextLoader("../data/text_files/python_intro.txt")
loader2 = TextLoader("../data/text_files/machine_learning.txt")

# PDF folder loader
pdf_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)


document1 = loader1.load()
document2 = loader2.load()
document3 = pdf_loader.load()

documents = document1 + document2 + document3

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

C:\Users\acer\AppData\Local\Temp\ipykernel_9040\1374224585.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\acer\AppData\Local\Temp\ipykernel_9040\1374224585.py:22: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 966.47it/s]


In [4]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": os.getenv("typesense_host"),  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key": os.getenv("typesense_api_key"),
        "typesense_collection_name": "lang-chain"
    },
    
)

In [5]:
query = "what is python"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.


In [6]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000001B8D3D95940>, search_kwargs={})

In [7]:
query = "What is Python"
retriever.invoke(query)[0]

Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')

In [8]:
from langchain_groq import ChatGroq

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,
             model_name="llama-3.1-8b-instant",
             temperature=0.1,max_tokens=1024)

# Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):

    # Retrieve context
    results = retriever.invoke(query)

    # Limit top_k results
    results = results[:top_k]

    # Combine retrieved chunks
    context = "\n\n".join([doc.page_content for doc in results])

    if not context:
        return "No relevant context found to answer the question."

    # Prompt
    prompt = f"""
    Use the following context to answer the question concisely.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Generate answer
    response = llm.invoke(prompt)

    return "Answer: " + response.content 

In [9]:
query = "what are the types of machine learning"

q1 = rag_simple(query, retriever, llm)

print(q1)

Answer: There are three types of machine learning:

1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties


In [10]:
query = "What is cloud computing"

q2 = rag_simple(query, retriever, llm)

print(q2)

Answer: Cloud computing refers to the delivery of various services over the internet, including data storage, servers, databases, networking, and software.
